# Train Falcon model using SageMaker SDK v3 ModelTrainer with FSx Lustre + FSDP

---

This notebook demonstrates how to train/fine-tune [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b) on the [GLUE/SST2](https://huggingface.co/datasets/glue/viewer/sst2/train) dataset using:

- **SageMaker Python SDK v3 `ModelTrainer` API** (replaces the legacy `PyTorch` Estimator)
- **FSx for Lustre** as the high-performance data source (synced with S3)
- **PyTorch FSDP** for distributed training across multiple GPUs
- **Torchrun** distribution via ModelTrainer's `distributed` config

## Prerequisites

1. Deploy the CloudFormation stack (`cfn-vpc-fsx-lustre.yaml`) to create VPC, subnet, security group, and FSx Lustre filesystem linked to your S3 bucket.
2. Upload your preprocessed training data to the linked S3 bucket so it syncs to FSx automatically.

## Files

* `scripts/train.py` — Training entry point with FSDP
* `scripts/utils.py` — Helper for dataloaders and model saving
* `scripts/requirements.txt` — Dependencies

## 1. Getting started

Install the SageMaker Python SDK v3 and dependencies.

In [ ]:
!pip install "sagemaker>=3.0" "transformers" "datasets[s3]" "boto3" --upgrade

In [ ]:
!pip install -r scripts/requirements.txt 

## 2. Session setup and CloudFormation stack outputs

Initialize the SageMaker session and retrieve FSx Lustre, networking, and S3 configuration from the deployed CloudFormation stack. Set `STACK_NAME` to match the name you used when deploying `cfn-vpc-fsx-lustre.yaml`.

In [ ]:
import os
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.utils.utils import SageMakerClient

os.environ["AWS_PROFILE"] = "claude"
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"
os.environ["AWS_DEFAULT_PROFILE"] = "claude"
os.environ.pop("AWS_SESSION_TOKEN", None)

# Reset the SDK's singleton client so it picks up the correct credentials
SageMakerClient.reset()

region = "us-west-2"
boto_session = boto3.Session(profile_name="claude", region_name=region)
sm_client = boto_session.client("sagemaker")
session = Session(boto_session=boto_session, sagemaker_client=sm_client)

print(f"SageMaker role ARN: {role}")
print(f"Region: {region}")

# --- Read CloudFormation stack outputs ---
STACK_NAME = "fsx-setup"  # <-- Change this to your CloudFormation stack name

cfn = boto_session.client("cloudformation")
outputs = cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["Outputs"]
stack_outputs = {o["OutputKey"]: o["OutputValue"] for o in outputs}

FSX_FILE_SYSTEM_ID = stack_outputs["FileSystemId"]
FSX_MOUNT_NAME = stack_outputs["MountName"]
SUBNET_ID = stack_outputs["SubnetId"]
SECURITY_GROUP_ID = stack_outputs["SecurityGroupId"]
S3_BUCKET = stack_outputs["S3BucketLinked"]

FSX_DIRECTORY_PATH = f"/{FSX_MOUNT_NAME}"

print(f"\nCloudFormation stack: {STACK_NAME}")
print(f"  S3 Bucket (FSx-linked): {S3_BUCKET}")
print(f"  FSx File System ID:     {FSX_FILE_SYSTEM_ID}")
print(f"  FSx Mount Name:         {FSX_MOUNT_NAME}")
print(f"  Subnet ID:              {SUBNET_ID}")
print(f"  Security Group ID:      {SECURITY_GROUP_ID}")
print(f"  Directory Path:         {FSX_DIRECTORY_PATH}")

In [ ]:
role

## 3. Load and prepare the dataset

We use the [GLUE/SST2](https://huggingface.co/datasets/glue/viewer/sst2/train) dataset, tokenize it, and chunk into 2048-token blocks.

In [ ]:
model_id = "tiiuae/falcon-7b"
dataset_name = "glue"
dataset_config = "sst2"

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)
dataset = load_dataset(dataset_name, dataset_config)
dataset = dataset.shuffle(42)

In [ ]:
if "validation" not in dataset.keys():
    dataset["validation"] = load_dataset(dataset_name, split="train[:5%]")
    dataset["train"] = load_dataset(dataset_name, split="train[5%:]")

In [ ]:
from itertools import chain
from functools import partial


def group_texts(examples, block_size=2048):
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result


column_names = dataset["train"].column_names
text_column_name = "text" if "text" in column_names else column_names[0]

lm_dataset = dataset.map(
    lambda sample: tokenizer(sample[text_column_name]),
    batched=True,
    remove_columns=list(column_names),
    desc="Running tokenizer on dataset",
).map(
    partial(group_texts, block_size=2048),
    batched=True,
)

## 4. Upload data to S3

Save the tokenized dataset to S3. If using FSx Lustre linked to this bucket, the data will be automatically available on the filesystem.

In [ ]:
s3_data_path = f"s3://{S3_BUCKET}/processed/data/"
storage_options = {"profile": "claude"}

print(f"Saving training dataset to: {s3_data_path}")
lm_dataset.save_to_disk(s3_data_path, storage_options=storage_options)
print(f"Uploaded data to: {s3_data_path}")

## 5. Configure and launch training with ModelTrainer

We use the SageMaker SDK v3 `ModelTrainer` API with:
- `FileSystemDataSource` for FSx Lustre input channel
- `Torchrun` distributed config for multi-GPU FSDP training
- `Networking` config for VPC/subnet/security group (must match FSx Lustre placement)

In [ ]:
import time
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import (
    SourceCode,
    Compute,
    Networking,
    InputData,
    OutputDataConfig,
    StoppingCondition,
    FileSystemDataSource,
)
from sagemaker.train.distributed import Torchrun

# --- Training image (PyTorch 2.0+ GPU DLC for your region) ---
training_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-training:2.0.1-gpu-py310-cu118-ubuntu20.04-sagemaker"

# --- Source code ---
source_code = SourceCode(
    source_dir="./scripts",
    entry_script="train.py",
    requirements="requirements.txt",
)

# --- Compute: 2x ml.g5.12xlarge (4 GPUs each) ---
compute = Compute(
    instance_type="ml.g5.2xlarge",
    instance_count=1,
    volume_size_in_gb=400,
)

# --- Networking: must match FSx Lustre VPC/subnet ---
networking = Networking(
    subnets=[SUBNET_ID],
    security_group_ids=[SECURITY_GROUP_ID],
    enable_network_isolation=False,
    enable_inter_container_traffic_encryption=True,
)

# --- Distributed: Torchrun with 4 GPUs per node ---
distributed = Torchrun(process_count_per_node=4)

# --- FSx Lustre data channel ---
fsx_train_input = InputData(
    channel_name="train",
    data_source=FileSystemDataSource(
        file_system_id=FSX_FILE_SYSTEM_ID,
        file_system_access_mode="ro",
        file_system_type="FSxLustre",
        directory_path=FSX_DIRECTORY_PATH,
    ),
)

# --- Hyperparameters ---
hyperparameters = {
    "model_id": model_id,
    "dataset_path": "/opt/ml/input/data/train",
    "gradient_checkpointing": True,
    "bf16": True,
    "optimizer": "adamw_torch",
    "per_device_train_batch_size": 1,
    "epochs": 1,
    "fsdp": "full_shard auto_wrap",
    "max_steps": 30,
}

# --- Output config ---
output_data_config = OutputDataConfig(
    s3_output_path=f"s3://{S3_BUCKET}/falcon-fsdp-output/",
)

# --- Stopping condition ---
stopping_condition = StoppingCondition(max_runtime_in_seconds=3600)

# --- Assemble ModelTrainer ---
model_trainer = ModelTrainer(
    training_image=training_image,
    source_code=source_code,
    distributed=distributed,
    compute=compute,
    networking=networking,
    hyperparameters=hyperparameters,
    input_data_config=[fsx_train_input],
    output_data_config=output_data_config,
    stopping_condition=stopping_condition,
    environment={"NCCL_DEBUG": "INFO"},
    role=role,
    sagemaker_session=session,
    base_job_name="falcon-fsdp-fsx",
)

print("ModelTrainer configured successfully.")
print(f"  Image: {training_image}")
print(f"  Compute: {compute.instance_count}x {compute.instance_type}")
print(f"  FSx filesystem: {FSX_FILE_SYSTEM_ID}")
print(f"  Distribution: Torchrun (4 procs/node)")

### Launch the training job

In [22]:
model_trainer.train(wait=True, logs=True)

## 6. Expected Output

After training begins, you should see output similar to:

```
4%|▍         | 1/25 [00:08<03:29,  8.72s/it]
...
100%|██████████| 25/25 [00:41<00:00,  1.65s/it]
******epoch=0: train_ppl=tensor(43260.7148, device='cuda:0') train_loss=tensor(10.6750, device='cuda:0')******
*******epoch=0: eval_ppl=tensor(nan, device='cuda:0') eval_loss=tensor(nan, device='cuda:0')*******
Training done!
```

## 7. (Alternative) Using S3 data source instead of FSx

If you don't have FSx Lustre set up, you can use an S3 data source directly:

In [ ]:
# Alternative: S3 data channel (no FSx required, no networking config needed)
# from sagemaker.core.training.configs import InputData
#
# s3_train_input = InputData(
#     channel_name="train",
#     data_source=s3_data_path,
# )
#
# model_trainer_s3 = ModelTrainer(
#     training_image=training_image,
#     source_code=source_code,
#     distributed=distributed,
#     compute=compute,
#     hyperparameters=hyperparameters,
#     input_data_config=[s3_train_input],
#     output_data_config=output_data_config,
#     stopping_condition=stopping_condition,
#     role=role,
#     sagemaker_session=session,
#     base_job_name="falcon-fsdp-s3",
# )
# model_trainer_s3.train(wait=True, logs=True)